# Výzva: Analýza textu o dátovej vede

V tomto príklade si urobíme jednoduché cvičenie, ktoré pokrýva všetky kroky tradičného procesu dátovej vedy. Nemusíte písať žiaden kód, stačí kliknúť na bunky nižšie, vykonať ich a pozorovať výsledok. Ako výzvu vám odporúčame skúsiť tento kód s rôznymi údajmi.

## Cieľ

V tejto lekcii sme hovorili o rôznych konceptoch súvisiacich s dátovou vedou. Skúsme objaviť viac súvisiacich konceptov pomocou **textového dolovania**. Začneme textom o dátovej vede, z neho vyextrahujeme kľúčové slová a potom sa pokúsime výsledok vizualizovať.

Ako text použijem stránku o dátovej vede z Wikipédie:


In [ ]:
url = 'https://en.wikipedia.org/wiki/Data_science'

## Krok 1: Získavanie údajov

Prvým krokom v každom procese dátovej vedy je získavanie údajov. Na to použijeme knižnicu `requests`:


In [ ]:
import requests

# Define a custom header.
headers = {
    'User-Agent': 'DataScienceChallenge/1.0 (myemail@gmail.com)'
}

# Pass the headers into the get request
response = requests.get(url, headers=headers)

if response.status_code == 200:
    text = response.content.decode('utf-8')
    print(text[:1000])
else:
    print(f"Error: {response.status_code}")

## Krok 2: Transformácia údajov

Ďalším krokom je previesť údaje do formy vhodnej na spracovanie. V našom prípade sme si stiahli zdrojový kód HTML zo stránky a potrebujeme ho previesť na obyčajný text.

Existuje veľa spôsobov, ako to urobiť. Použijeme [BeautifulSoup](https://www.crummy.com/software/BeautifulSoup/), populárnu knižnicu v Pythone na parsovanie HTML. BeautifulSoup nám umožňuje zamerať sa na konkrétne HTML prvky, takže sa môžeme sústrediť na hlavný obsah článku z Wikipédie a zredukovať niektoré navigačné menu, postranné panely, päty a iný nerelevantný obsah (hoci niektorý boilerplate text môže zostať).


Najprv potrebujeme nainštalovať knižnicu BeautifulSoup na analýzu HTML:


In [ ]:
import sys
!{sys.executable} -m pip install beautifulsoup4

In [ ]:
from bs4 import BeautifulSoup

# Parse the HTML content
soup = BeautifulSoup(text, 'html.parser')

# Extract only the main article content from Wikipedia
# Wikipedia uses 'mw-parser-output' class for the main article content
content = soup.find('div', class_='mw-parser-output')

def clean_wikipedia_content(content_node):
    """Remove common non-article elements from a Wikipedia content node."""
    # Strip jump links, navboxes, reference lists/superscripts, edit sections, TOC, sidebars, etc.
    selectors = [
        '.mw-jump-link',
        '.navbox',
        '.reflist',
        'sup.reference',
        '.mw-editsection',
        '.hatnote',
        '.metadata',
        '.infobox',
        '#toc',
        '.toc',
        '.sidebar',
    ]
    for selector in selectors:
        for el in content_node.select(selector):
            el.decompose()

if content:
    # Clean the content node to better approximate article text only.
    clean_wikipedia_content(content)
    text = content.get_text(separator=' ', strip=True)
    print(text[:1000])
else:
    print("Could not find main content. Using full page text.")
    text = soup.get_text(separator=' ', strip=True)
    print(text[:1000])

## Krok 3: Získavanie poznatkov

Najdôležitejším krokom je premeniť naše údaje na nejakú formu, z ktorej môžeme čerpať poznatky. V našom prípade chceme z textu extrahovať kľúčové slová a zistiť, ktoré kľúčové slová sú významnejšie.

Použijeme Python knižnicu nazvanú [RAKE](https://github.com/aneesha/RAKE) na extrakciu kľúčových slov. Najskôr si túto knižnicu nainštalujeme, ak ešte nie je prítomná: 


In [ ]:
import sys
!{sys.executable} -m pip install nlp_rake

Hlavná funkcionalita je dostupná z objektu `Rake`, ktorý môžeme prispôsobiť pomocou niektorých parametrov. V našom prípade nastavíme minimálnu dĺžku kľúčového slova na 5 znakov, minimálnu frekvenciu kľúčového slova v dokumente na 3 a maximálny počet slov v kľúčovom slove na 2. Neváhajte si pohrávať s ďalšími hodnotami a pozorovať výsledok.


In [ ]:
import nlp_rake
extractor = nlp_rake.Rake(max_words=2,min_freq=3,min_chars=5)
res = extractor.apply(text)
res


Získali sme zoznam pojmov spolu s priradeným stupňom dôležitosti. Ako vidíte, najrelevantnejšie disciplíny, ako napríklad strojové učenie a veľké dáta, sa nachádzajú v zozname na popredných pozíciách.

## Krok 4: Vizualizácia výsledku

Ľudia dokážu najlepšie interpretovať údaje vo vizuálnej forme. Preto často dáva zmysel údaje vizualizovať, aby sme získali určité poznatky. Na jednoduché zobrazenie distribúcie kľúčových slov s ich relevantnosťou môžeme použiť knižnicu `matplotlib` v Pythone:


In [ ]:
import matplotlib.pyplot as plt

def plot(pair_list):
    k,v = zip(*pair_list)
    plt.bar(range(len(k)),v)
    plt.xticks(range(len(k)),k,rotation='vertical')
    plt.show()

plot(res)

Existuje však ešte lepší spôsob, ako vizualizovať frekvenciu slov – pomocou **Word Cloud**. Budeme si musieť nainštalovať ďalšiu knižnicu na vykreslenie word cloudu z nášho zoznamu kľúčových slov.


In [ ]:
!{sys.executable} -m pip install wordcloud

`WordCloud` objekt je zodpovedný za prijatie buď pôvodného textu, alebo predpočítaného zoznamu slov s ich frekvenciami, a vracia obrázok, ktorý môže byť následne zobrazený pomocou `matplotlib`:


In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

wc = WordCloud(background_color='white',width=800,height=600)
plt.figure(figsize=(15,7))
plt.imshow(wc.generate_from_frequencies({ k:v for k,v in res }))

Pôvodný text môžeme tiež odovzdať funkcii `WordCloud` - uvidíme, či dokážeme dosiahnuť podobný výsledok:


In [ ]:
plt.figure(figsize=(15,7))
plt.imshow(wc.generate(text))

In [ ]:
wc.generate(text).to_file('images/ds_wordcloud.png')

Teraz vidíte, že slovný mrak vyzerá pôsobivejšie, ale obsahuje aj veľa šumu (napr. nesúvisiace slová ako `Retrieved on`). Tiež získavame menej kľúčových slov, ktoré pozostávajú z dvoch slov, napríklad *data scientist* alebo *computer science*. Je to preto, že algoritmus RAKE lepšie vyberá dobré kľúčové slová z textu. Tento príklad ilustruje dôležitosť predspracovania a čistenia dát, pretože jasný obraz na konci nám umožní robiť lepšie rozhodnutia.

V tomto cvičení sme prešli jednoduchým procesom extrahovania významu z textu Wikipédie vo forme kľúčových slov a slovného mraku. Tento príklad je dosť jednoduchý, ale dobre demonštruje všetky typické kroky, ktoré data scientist vykonáva pri práci s dátami, od ich získania až po vizualizáciu.

V našom kurze podrobne preberieme všetky tieto kroky. 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
